# 07b -- Features extendidas (fase B, parte 1)

Construye `matriz_as_of_v2.parquet`: **las mismas filas** de `matriz_as_of.parquet` (07) más features
que el modelo baseline no usa. La ablación (¿mejoran algo?) está en `07c`.

Lo que se agrega, todo as-of (solo información hasta el origen `t`, incluido):

| Grupo | Features | Idea |
|---|---|---|
| Memoria corta/larga | `trail_7`, `trail_28`, `ewma_14`, `ewma_28`, `dias_con_venta_15` | el baseline solo ve 15 y 60 días |
| Momentum relativo | `ratio_trail_nivel` = demanda 15d / (nivel 60d x H) | una razón no depende de la escala del producto |
| Contexto de categoría | `cat_ratio` (tendencia categoría x sucursal), `share_cat` (peso del producto) | información que el producto solo no trae |
| Contexto de sucursal | `suc_ratio` (tendencia de la sucursal) | demanda agregada como señal |
| Calendario de la ventana | `n_festivos_ventana`, `n_inicio_mes_ventana` | el calendario futuro es conocido, no es fuga |

Quedan fuera a propósito: la interacción precio (el 04 mostró efecto contrario al esperado) y el
año anterior (nov-dic 2022 no existe, dejaría NaN sistemáticos).

Cadena: ... -> `07_matriz_as_of` -> **`07b_features_extendidas`** -> `07c_ablacion_features`

In [1]:
import sys, os
sys.path.insert(0, '.')
from common_priorizacion import (DW, registrar_huella, verificar_huella,
                                 leer_panel, leer_calendario_habil, leer_dim_priorizado)
import evaluar_nivel1 as ev

import json
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

HORIZONTE = int(os.environ.get('HORIZONTE_07', 15))
SUF = '' if HORIZONTE == 15 else f'_h{HORIZONTE}'
EXPERIMENTO = 'EDA-fix-nivel1-07b-features-extendidas'

verificar_huella(f'huella_07{SUF}.json', [f'matriz_as_of{SUF}.parquet'])
matriz = pd.read_parquet(f'{DW}/matriz_as_of{SUF}.parquet')
diario = leer_panel()
fechas_habiles, indice_habil = leer_calendario_habil()
dim = leer_dim_priorizado()
calendario_eventos = pd.read_csv(f'{DW}/calendario_eventos.csv', parse_dates=['fecha'])
print(f'matriz base: {matriz.shape} | horizonte {HORIZONTE}')

Huella verificada OK contra huella_07.json.
matriz base: (262429, 27) | horizonte 15


## 1. Rejilla hábil y grupos de contexto

Misma construcción que 07 (días hábiles x pares, ausencia de venta = 0 explícito).

In [2]:
ancha = (diario.pivot_table(index='fecha', columns=['codigo_item', 'sucursal'],
                            values='unidades', aggfunc='sum', fill_value=0)
         .reindex(fechas_habiles, fill_value=0)
         .astype('float32'))
pares = pd.MultiIndex.from_arrays([ancha.columns.get_level_values(0).astype(str),
                                   ancha.columns.get_level_values(1).astype(str)])
assert ancha.index.equals(pd.DatetimeIndex(fechas_habiles))

cat_de = dim.set_index('codigo_item')['categoria'].astype(str).to_dict()
categoria = np.array([cat_de.get(i, 'SIN_CATEGORIA') for i in pares.get_level_values(0)])
suc = pares.get_level_values(1).to_numpy()
gid_cat = pd.factorize(pd.Series(categoria + '|' + suc))[0]      # grupo categoría x sucursal
gid_suc = pd.factorize(pd.Series(suc))[0]                        # grupo sucursal
print(f'rejilla {ancha.shape} | grupos categoría x sucursal: {gid_cat.max()+1} | sucursales: {gid_suc.max()+1}')

rejilla (1362, 11135) | grupos categoría x sucursal: 96 | sucursales: 3


## 2. Features vectorizadas (solo causales)

Todo son `rolling`/`ewm` hacia atrás sobre la rejilla, así que la fila de `t` solo depende de días
`<= t`. Eso se **verifica** en la sección 4, no se asume.

In [3]:
H = HORIZONTE
EPS = 1e-6

def _suma_grupo_rolling(A, gid, ventana):
    """Suma por grupo (columnas) de la demanda acumulada `ventana` días, devuelta por columna original."""
    G = A.T.groupby(gid).sum().T                      # T x n_grupos
    r = G.rolling(ventana, min_periods=ventana).sum()
    return r, r.shift(ventana)

def grids_v2(A):
    """A: DataFrame (T x P). Devuelve {feature: ndarray T x P}. Causal por construcción."""
    out = {}
    t_h = A.rolling(H, min_periods=H).sum()
    n60 = A.rolling(60, min_periods=20).mean()
    out['trail_7'] = A.rolling(7, min_periods=7).sum().to_numpy()
    out['trail_28'] = A.rolling(28, min_periods=28).sum().to_numpy()
    out['ewma_14'] = A.ewm(span=14, adjust=False).mean().to_numpy()
    out['ewma_28'] = A.ewm(span=28, adjust=False).mean().to_numpy()
    out['dias_con_venta_15'] = (A > 0).astype('float32').rolling(15, min_periods=15).sum().to_numpy()
    out['ratio_trail_nivel'] = (t_h / (n60 * H + EPS)).to_numpy()
    for nombre, gid in (('cat', gid_cat), ('suc', gid_suc)):
        g_t, g_prev = _suma_grupo_rolling(A, gid, H)
        out[f'{nombre}_ratio'] = ((g_t + 1.0) / (g_prev + 1.0)).loc[:, gid].to_numpy()   # suavizado: con g_prev=0 la razón cruda explotaba (2e6)
        if nombre == 'cat':
            out['share_cat'] = (t_h.to_numpy() / (g_t.loc[:, gid].to_numpy() + EPS))
    return {k: v.astype('float32') for k, v in out.items()}

G2 = grids_v2(ancha)
print({k: v.shape for k, v in G2.items()})

{'trail_7': (1362, 11135), 'trail_28': (1362, 11135), 'ewma_14': (1362, 11135), 'ewma_28': (1362, 11135), 'dias_con_venta_15': (1362, 11135), 'ratio_trail_nivel': (1362, 11135), 'cat_ratio': (1362, 11135), 'share_cat': (1362, 11135), 'suc_ratio': (1362, 11135)}


In [4]:
# Calendario de la ventana de pronóstico t+1..t+H (el calendario futuro es conocido: no es fuga)
cal = calendario_eventos.set_index('fecha').reindex(fechas_habiles)
fest = cal['es_festivo'].fillna(False).astype(float)
inicio = (cal['dia'] <= 3).astype(float)
n_fest = fest.rolling(H, min_periods=H).sum().shift(-H)
n_inicio = inicio.rolling(H, min_periods=H).sum().shift(-H)

# Lookup fila x columna para cada fila de la matriz
ridx = matriz['fecha_origen'].map(indice_habil).to_numpy()
assert not np.isnan(ridx.astype(float)).any(), 'hay orígenes fuera del calendario hábil'
ridx = ridx.astype(int)
llaves = pd.MultiIndex.from_arrays([matriz['codigo_item'].astype(str), matriz['sucursal'].astype(str)])
cidx = pares.get_indexer(llaves)
assert (cidx >= 0).all(), 'hay pares de la matriz que no están en la rejilla'

v2 = matriz.copy()
for nombre, grid in G2.items():
    v2[nombre] = grid[ridx, cidx]
v2['n_festivos_ventana'] = n_fest.to_numpy()[ridx]
v2['n_inicio_mes_ventana'] = n_inicio.to_numpy()[ridx]

NUEVAS = list(G2.keys()) + ['n_festivos_ventana', 'n_inicio_mes_ventana']
print(f'{len(NUEVAS)} features nuevas:', NUEVAS)
print(v2[NUEVAS].describe().T[['count', 'mean', 'min', '50%', 'max']])

11 features nuevas: ['trail_7', 'trail_28', 'ewma_14', 'ewma_28', 'dias_con_venta_15', 'ratio_trail_nivel', 'cat_ratio', 'share_cat', 'suc_ratio', 'n_festivos_ventana', 'n_inicio_mes_ventana']


                           count   mean   min   50%       max
trail_7              262,429.000  4.344 0.000 1.000 2,569.000
trail_28             262,429.000 17.807 0.000 5.000 9,113.000
ewma_14              262,429.000  0.630 0.000 0.151   338.757
ewma_28              262,429.000  0.632 0.000 0.162   327.658
dias_con_venta_15    262,429.000  3.156 0.000 2.000    15.000
ratio_trail_nivel    262,429.000  0.892 0.000 0.832     4.000
cat_ratio            262,429.000  1.027 0.179 0.983    10.226
share_cat            262,429.000  0.015 0.000 0.003     1.000
suc_ratio            262,429.000  1.011 0.605 0.988     1.454
n_festivos_ventana   262,429.000  0.709 0.000 1.000     2.000
n_inicio_mes_ventana 262,429.000  1.552 0.000 2.000     3.000


## 3. Calidad: nulos e infinitos

Sin `fillna` silencioso: si una feature trae nulos se investiga, no se rellena.

In [5]:
nulos = v2[NUEVAS].isna().sum()
print(nulos[nulos > 0] if (nulos > 0).any() else 'sin nulos en las features nuevas')
assert not (nulos > 0).any(), f'features con nulos: {nulos[nulos > 0].to_dict()} -- revisar min_periods/ventanas'
assert np.isfinite(v2[NUEVAS].to_numpy()).all(), 'hay infinitos en las features nuevas'
razones = ['ratio_trail_nivel', 'cat_ratio', 'suc_ratio']
assert v2[razones].max().max() < 100, f'razón fuera de escala (un GLM se rompería): {v2[razones].max().to_dict()}'
print('razones acotadas:', v2[razones].max().round(2).to_dict())

sin nulos en las features nuevas
razones acotadas: {'ratio_trail_nivel': 4.0, 'cat_ratio': 10.229999542236328, 'suc_ratio': 1.4500000476837158}


## 4. Prueba de no-fuga

Para 4 orígenes de la matriz se recalculan las features usando **solo** la historia `<= t` (rejilla
truncada) y se comparan con las de la rejilla completa. Si alguna feature dependiera de días
posteriores, difieren.

In [6]:
_ori = sorted(matriz['fecha_origen'].unique())
orig_prueba = [_ori[j] for j in (0, len(_ori) // 3, 2 * len(_ori) // 3, len(_ori) - 1)]
max_dif = {}
for o in orig_prueba:
    i = indice_habil[pd.Timestamp(o)]
    Gt = grids_v2(ancha.iloc[:i + 1])
    for k, g in Gt.items():
        completo, truncado = G2[k][i], g[-1]
        ok = np.isclose(completo, truncado, rtol=1e-4, atol=1e-4, equal_nan=True)
        max_dif[k] = max(max_dif.get(k, 0.0), float(np.nanmax(np.abs(completo - truncado))))
        assert ok.all(), f'FUGA en {k} para el origen {pd.Timestamp(o).date()}: {(~ok).sum()} pares difieren'
print('sin fuga temporal: features idénticas con la rejilla truncada en', [str(pd.Timestamp(o).date()) for o in orig_prueba])
print('máxima diferencia observada por feature:', {k: round(v, 6) for k, v in max_dif.items()})

sin fuga temporal: features idénticas con la rejilla truncada en ['2024-03-25', '2024-10-30', '2025-05-18', '2025-12-12']
máxima diferencia observada por feature: {'trail_7': 0.0, 'trail_28': 0.0, 'ewma_14': 0.0, 'ewma_28': 0.0, 'dias_con_venta_15': 0.0, 'ratio_trail_nivel': 0.0, 'cat_ratio': 0.0, 'share_cat': 0.0, 'suc_ratio': 0.0}


In [7]:
# Las columnas originales no cambian y las filas son las mismas
base = pd.read_parquet(f'{DW}/matriz_as_of{SUF}.parquet')
assert len(v2) == len(base) and list(v2.columns[:len(base.columns)]) == list(base.columns)
assert v2[base.columns].equals(base), 'las columnas de 07 fueron alteradas'
print('columnas de 07 intactas, mismas filas:', v2.shape)

v2.to_parquet(f'{DW}/matriz_as_of_v2{SUF}.parquet', index=False)
huella = {'notebook': '07b_features_extendidas', 'fecha_generacion': pd.Timestamp.now().isoformat(),
          'parametros': {'HORIZONTE': HORIZONTE, 'features_nuevas': NUEVAS},
          'entradas': registrar_huella([f'matriz_as_of{SUF}.parquet', 'panel_diario.parquet', 'calendario_habil.csv',
                                        'calendario_eventos.csv', 'dim_producto_priorizado.parquet']),
          'salidas': registrar_huella([f'matriz_as_of_v2{SUF}.parquet'])}
with open(f'{DW}/huella_07b{SUF}.json', 'w') as f:
    json.dump(huella, f, indent=2, ensure_ascii=False)
print(f'guardado: matriz_as_of_v2{SUF}.parquet, huella_07b{SUF}.json')

columnas de 07 intactas, mismas filas: (262429, 38)


guardado: matriz_as_of_v2.parquet, huella_07b.json


## 5. Registro en MLflow

In [8]:
run_id = ev.registrar_mlflow(
    EXPERIMENTO, f'features_extendidas_h{HORIZONTE}',
    tags={'fase': 'B', 'notebook': '07b', 'rama_git': 'fix/eda-modelado-nivel1'},
    params={'HORIZONTE': HORIZONTE, 'n_features_nuevas': len(NUEVAS), 'features_nuevas': ','.join(NUEVAS),
            'huella_matriz_v2': huella['salidas'][f'matriz_as_of_v2{SUF}.parquet']},
    metrics={'n_filas': len(v2), 'n_columnas': v2.shape[1], 'max_dif_prueba_fuga': max(max_dif.values())})
print('run', run_id)

/home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/24 17:36:13 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


🏃 View run features_extendidas_h15 at: http://127.0.0.1:5000/#/experiments/15/runs/8d5206e58e574635b50ead5d706c6df6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/15
run 8d5206e58e574635b50ead5d706c6df6
